<center>
<img src="https://laelgelcpublic.s3.sa-east-1.amazonaws.com/lael_50_years_narrow_white.png.no_years.400px_96dpi.png" width="300" alt="LAEL 50 years logo">
<h3>APPLIED LINGUISTICS GRADUATE PROGRAMME (LAEL)</h3>
</center>
<hr>

## Path Constants

In [1]:
PROJECT = "cl_st2_ph2_arianne"

In [2]:
from pathlib import Path

SCORES_FILE = Path(f"sas/output_{PROJECT}/{PROJECT}_scores_only.tsv")
MEANS_PATTERN = f"sas/output_{PROJECT}/means_group_f{{dim}}.tsv"
FILE_IDS_PATH = Path("file_ids.txt")

## Load factor scores

In [3]:
import pandas as pd

scores_df = pd.read_csv(SCORES_FILE, sep="\t")
scores_df = scores_df.rename(columns={"filename": "file id"})

scores_df

,file id,source,model,prompt,group,fac1,fac2,fac3,fac4,v000001,...,v000961,v000962,v000963,v000964,v000965,v000966,v000967,v000968,v000969,v000970
0,t000001,ai,gemini,persona,persona_gemini,7,6,0,2,0,...,1,0,0,0,0,0,0,0,0,0
1,t000002,ai,gemini,persona,persona_gemini,3,7,5,1,0,...,1,0,0,0,0,0,0,0,0,0
2,t000003,ai,gemini,persona,persona_gemini,10,8,0,0,0,...,1,0,0,0,1,0,0,0,0,0
3,t000004,ai,gemini,persona,persona_gemini,4,7,6,0,0,...,1,0,0,0,0,0,0,0,0,0
4,t000005,ai,gemini,persona,persona_gemini,5,7,5,3,0,...,0,1,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,t003996,human,human,human,human,82,21,8,1,0,...,1,1,1,1,1,0,1,0,0,0
3996,t003997,human,human,human,human,46,12,3,1,0,...,1,1,1,0,1,0,0,0,0,1
3997,t003998,human,human,human,human,39,8,4,3,0,...,1,0,0,0,1,0,0,0,0,0
3998,t003999,human,human,human,human,99,22,21,14,0,...,1,0,1,0,1,0,0,0,0,0


## Load file ID mapping

In [4]:
file_ids_df = pd.read_csv(
    FILE_IDS_PATH,
    sep=" ",
    names=["file id", "group_filename"],
)

file_ids_df.head()

,file id,group_filename
0,t000001,t001_gemini.txt
1,t000002,t002_gemini.txt
2,t000003,t003_gemini.txt
3,t000004,t004_gemini.txt
4,t000005,t005_gemini.txt


## Merge factor scores with file ID mapping

In [5]:
scores_file_ids_df = scores_df.merge(file_ids_df, on="file id", how="left")
scores_file_ids_df = scores_file_ids_df[
    ["file id", "group_filename"]
    + [col for col in scores_file_ids_df.columns if col not in ["file id", "group_filename"]]
    ]

scores_file_ids_df


,file id,group_filename,source,model,prompt,group,fac1,fac2,fac3,fac4,...,v000961,v000962,v000963,v000964,v000965,v000966,v000967,v000968,v000969,v000970
0,t000001,t001_gemini.txt,ai,gemini,persona,persona_gemini,7,6,0,2,...,1,0,0,0,0,0,0,0,0,0
1,t000002,t002_gemini.txt,ai,gemini,persona,persona_gemini,3,7,5,1,...,1,0,0,0,0,0,0,0,0,0
2,t000003,t003_gemini.txt,ai,gemini,persona,persona_gemini,10,8,0,0,...,1,0,0,0,1,0,0,0,0,0
3,t000004,t004_gemini.txt,ai,gemini,persona,persona_gemini,4,7,6,0,...,1,0,0,0,0,0,0,0,0,0
4,t000005,t005_gemini.txt,ai,gemini,persona,persona_gemini,5,7,5,3,...,0,1,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,t003996,t995_human.txt,human,human,human,human,82,21,8,1,...,1,1,1,1,1,0,1,0,0,0
3996,t003997,t996_human.txt,human,human,human,human,46,12,3,1,...,1,1,1,0,1,0,0,0,0,1
3997,t003998,t997_human.txt,human,human,human,human,39,8,4,3,...,1,0,0,0,1,0,0,0,0,0
3998,t003999,t998_human.txt,human,human,human,human,99,22,21,14,...,1,0,1,0,1,0,0,0,0,0


## Load group means

In [6]:
import re

factor_cols = [col for col in scores_file_ids_df.columns if re.fullmatch(r"fac\d+", col)]
factor_cols = sorted(factor_cols, key=lambda col: int(col.replace("fac", "")))

means_dfs = {}

for factor_col in factor_cols:
    fac_num = int(factor_col.replace("fac", ""))
    means_file = Path(MEANS_PATTERN.format(dim=fac_num))

    if not means_file.exists():
        raise FileNotFoundError(f"Means file not found: {means_file}")

    means_df = pd.read_csv(means_file, sep="\t")

    mean_col = f"Mean fac{fac_num}"

    if "group" not in means_df.columns:
        raise ValueError(f"'group' column missing from means file: {means_file}")

    if mean_col not in means_df.columns:
        raise ValueError(f"'{mean_col}' column missing from means file: {means_file}")

    means_df["group"] = means_df["group"].astype(str).str.strip()

    means_dfs[factor_col] = means_df

means_dfs["fac1"].head()

,Effect,group,N,Mean fac1,SD fac1
0,group,human,1000,24.693,14.953540
1,group,persona_gemini,1000,3.998,2.932677
2,group,persona_gpt,1000,12.108,7.512175
3,group,persona_grok,1000,4.569,3.391274


In [7]:
means_dfs

{'fac1':   Effect           group     N  Mean fac1    SD fac1
 0  group           human  1000     24.693  14.953540
 1  group  persona_gemini  1000      3.998   2.932677
 2  group     persona_gpt  1000     12.108   7.512175
 3  group    persona_grok  1000      4.569   3.391274,
 'fac2':   Effect           group     N  Mean fac2    SD fac2
 0  group           human  1000     20.725  11.471660
 1  group  persona_gemini  1000      7.991   4.293874
 2  group     persona_gpt  1000     25.534  12.232354
 3  group    persona_grok  1000      9.008   5.235776,
 'fac3':   Effect           group     N  Mean fac3   SD fac3
 0  group           human  1000     14.831  9.920830
 1  group  persona_gemini  1000      5.007  3.917435
 2  group     persona_gpt  1000     11.757  7.446435
 3  group    persona_grok  1000      5.039  3.955898,
 'fac4':   Effect           group     N  Mean fac4   SD fac4
 0  group           human  1000      8.185  6.647327
 1  group  persona_gemini  1000      2.914  3.321391
 

## Simulate selection process

### List factors

In [8]:
factor_cols = [col for col in scores_file_ids_df.columns if re.fullmatch(r"fac\d+", col)]
factor_cols = sorted(factor_cols, key=lambda col: int(col.replace("fac", "")))

print("\n".join(
    f"{'' if i == 0 else '#'}FACTOR = {factor!r}"
    for i, factor in enumerate(factor_cols)
))

FACTOR = 'fac1'
#FACTOR = 'fac2'
#FACTOR = 'fac3'
#FACTOR = 'fac4'


### Copy/Paste the factors list and Uncomment/Comment factors and poles accordingly

In [9]:
FACTOR = 'fac1'
#FACTOR = 'fac2'
#FACTOR = 'fac3'
#FACTOR = 'fac4'

#POLE = "positive"
POLE = "negative"

sorted_means_dfs = means_dfs[FACTOR].sort_values(f"Mean {FACTOR}", ascending=POLE == "negative")

display(sorted_means_dfs)

groups_in_order = sorted_means_dfs["group"].tolist()

print("\n".join(
    f"{'' if i == 0 else '#'}GROUP = {group!r}"
    for i, group in enumerate(groups_in_order)
))

,Effect,group,N,Mean fac1,SD fac1
1,group,persona_gemini,1000,3.998,2.932677
3,group,persona_grok,1000,4.569,3.391274
2,group,persona_gpt,1000,12.108,7.512175
0,group,human,1000,24.693,14.953540


GROUP = 'persona_gemini'
#GROUP = 'persona_grok'
#GROUP = 'persona_gpt'
#GROUP = 'human'


### Copy/Paste the groups list and Uncomment/Comment groups accordingly

In [10]:
GROUP = 'human'
#GROUP = 'persona_gpt'
#GROUP = 'persona_grok'
#GROUP = 'persona_gemini'

N_EXAMPLES = 25 # Define the number of examples to select

scores_file_ids_df.loc[scores_file_ids_df["group"].eq(GROUP)].nlargest(N_EXAMPLES, FACTOR)

,file id,group_filename,source,model,prompt,group,fac1,fac2,fac3,fac4,...,v000961,v000962,v000963,v000964,v000965,v000966,v000967,v000968,v000969,v000970
3451,t003452,t451_human.txt,human,human,human,human,101,22,14,14,...,1,0,1,1,1,0,0,0,0,0
3280,t003281,t280_human.txt,human,human,human,human,99,27,15,20,...,1,1,1,1,1,0,0,0,0,1
3998,t003999,t998_human.txt,human,human,human,human,99,22,21,14,...,1,0,1,0,1,0,0,0,0,0
3022,t003023,t023_human.txt,human,human,human,human,92,24,8,6,...,1,0,1,1,1,0,0,0,0,0
3226,t003227,t226_human.txt,human,human,human,human,91,51,29,16,...,1,0,1,0,1,0,1,0,0,0
3999,t004000,t999_human.txt,human,human,human,human,91,25,19,7,...,0,0,1,1,1,0,1,0,0,1
3272,t003273,t272_human.txt,human,human,human,human,88,23,10,22,...,1,0,1,1,1,0,0,0,0,0
3297,t003298,t297_human.txt,human,human,human,human,88,42,19,20,...,1,0,1,1,1,0,0,0,0,0
3185,t003186,t185_human.txt,human,human,human,human,85,71,26,12,...,1,1,1,0,1,0,0,0,0,0
3300,t003301,t300_human.txt,human,human,human,human,85,24,12,16,...,1,1,1,1,1,0,0,0,0,0
